In [3]:
import logging
import tensorflow as tf
from anndata import read_h5ad
from scaden.model.functions import get_signature_genes, preprocess_h5ad_data, custom_preprocess_h5ad_data
logger = logging.getLogger(__name__)

from concurrent.futures import ProcessPoolExecutor
from testingPurityFunctions.sharedConfig import generate_batches, BULK, BULK_PURITY, PURITY
# from logBatchesNew import load_completed_batches, mark_batch_completed
import scaden
print(scaden.__file__)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1' 
from scaden.simulate import simulation
from scaden.process import processing
from scaden.train import training
from scaden.predict import prediction


BASE_PATH = "/Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions/results/0_scadenInput/"
BULK = os.path.join(BASE_PATH, "01_Deconvolution_customScadenDFT1_bulkCountsFinalHVGs.txt")
BULK_PURITY = os.path.join(BASE_PATH, "01_Deconvolution_customScadenDFT1_bulkCountsFinalHVGs_withPurities_bin10.txt")
PURITY = "DFT1_cells"

/Users/nv4/gitClones/scadenPuritiesFullyRandomSim/scaden/__init__.py


In [ ]:
# ran = percRandom (mixed sample sampling), value between 0-1
# cran = celltypeRandom (how many of the samples should have a random sampling strategy), value between 0-1

subDir, bal, thr, ran, var, sam, cel, seed, lr, steps, cran = "/Users/nv4/gitClones/scadenPurities/testingPurityFunctions/results/0_scadenInput/snCounts/", False, 0, 0.5, 0.1, 20, 100, 123, 0.01, 10,  0.50
pfix = f"{os.path.basename(subDir.rstrip('/'))}_bal{bal}_thresh{thr}_ran{ran}_var{var}_samp{sam}_cells{cel}_seed{seed}_lr{lr}_steps{steps}_cran{cran}"
simDir = f"/Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions/results/1_simulate/{pfix}_test/"
    
os.makedirs(simDir, exist_ok=True)

simulation(
        simulate_dir=simDir,
        data_dir=subDir,
        sample_size=cel,
        num_samples=sam,
        pattern="*_counts.txt",
        unknown_celltypes=["unknown"],
        out_prefix=pfix,
        fmt="txt",
        balance=bal,
        threshold=thr,
        percRandom=ran,
        saveProp=True,
        seed=seed,
        remMerged=True,
        purity = PURITY, 
        cran = cran
    )

Random - multinomial sampler. 

In [ ]:
import numpy as np
import pandas as pd

threshold = 0

sample_size = 100 # number of cells

# celltypes_global = ["a", "b", "c", "d", "e"]

celltypes_global = []

sub_x = pd.DataFrame(
    np.random.rand(6, 10),
    columns=[f"Gene_{i}" for i in range(10)]
)

sub_y = pd.DataFrame({
    'Celltype': ['F', 'E', 'D', 'C', 'B', 'A']
})

available_celltypes = sub_y['Celltype'].unique().tolist()

# print(available_celltypes)
# print(sub_x)
# print(sub_y)

celltypes_global = sorted(set(sub_y['Celltype']))
# print(celltypes_global)

celltype_to_indices = {ct: sub_y[sub_y['Celltype'] == ct].index for ct in available_celltypes}
# print(f"Celltype to indices {celltype_to_indices}")

# Each cell type selected with equal randomness (1/num_celltypes)
random_multi_counts = np.random.multinomial(sample_size, [1/len(available_celltypes)] * len(available_celltypes))
print(f"Random counts {random_multi_counts}")

# random_int_counts = np.random.rand(sample_size, [1/len(available_celltypes)] * len(available_celltypes))
# print(f"Random counts {random_int_counts}")

celltype_order = np.random.permutation(available_celltypes)
print(f"Cell type order {celltype_order}")

sample_cells = []
fracs_complete = [0] * len(celltypes_global)
abs_counts = [0] * len(celltypes_global)
                        

Or randomly permute, but then you still have a skew towards DFT1 cells. Although this should reflect the training data. BUT it won't be the exact same proportions in each sample (as with balanced. So while average proportions may reflect balance = T, the variation in composition should differ.)

In [ ]:
mask = sub_y['Celltype'].isin(available_celltypes)
# print(mask)
combined = pd.concat([sub_x[mask], sub_y[mask]], axis=1)
shuffled = combined.sample(frac=1, replace=True).reset_index(drop=True)
sampled = shuffled.iloc[:sample_size]
# print(sampled)

sampled_cells = sampled.iloc[:, :-1]
# print(sampled_cells)
sampled_labels = sampled['Celltype']
print(sampled_labels)

fracs_complete = [0] * len(celltypes_global)
abs_counts = [0] * len(celltypes_global)

ct_counts = sampled_labels.value_counts()
for ct, count in ct_counts.items():
        idx = celltypes_global.index(ct)
        abs_counts[idx] = count
        fracs_complete[idx] = count / sample_size

df_samp = sampled_cells.sum(axis=0)

print(sampled_cells.head)

Or SMOTE?/With or without dropping a cell type each time?

In [5]:
subDir, bal, thr, ran, var, sam, cel, seed, lr, steps, cran = "/Users/nv4/gitClones/scadenPurities/testingPurityFunctions/results/0_scadenInput/snCounts/", False, 0, 0.5, 0.1, 20, 100, 123, 0.01, 10,  0.50
pfix = f"{os.path.basename(subDir.rstrip('/'))}_bal{bal}_thresh{thr}_ran{ran}_var{var}_samp{sam}_cells{cel}_seed{seed}_lr{lr}_steps{steps}_cran{cran}"
simDir = f"/Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions/results/1_simulate/{pfix}_test/"

# simDir = f"/Users/nv4/gitClones/scadenPurities/testingPurityFunctions/results/1_simulate/{pfix}_test/"
procDir = f"/Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions//results/2_process/{pfix}_test/"
os.makedirs(procDir, exist_ok=True)

training_data = os.path.join(simDir, pfix + ".h5ad")
processed_path = os.path.join(procDir, pfix + "_Processed.h5ad")

processing(
        data_path=BULK,
        training_data=training_data,
        processed_path=processed_path,
        var_cutoff=var ,
        ignore_genes=None, # Helpful if we need to ignore
        purity_col = "purity"#,
        # purity_bin_size=10, # Default. Should only bin IF there is a purity column in the simulated data. 
        # ignore_genes="purity" # Because this hasn't been set as a "gene" for the simulated data yet, we need to ignore it from the bulk matrix. Will adjust this so that it is done in this function iso in R before. 
)

Sig_genes_complete: 15809
Sig_genes: 2917
Process with purity input in X.obs. Not used as a sig gene.


In [6]:
print(steps)
print(lr)

10
0.01


In [7]:
modDir = f"/Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions/results/3_train/{pfix}/"
metDir = f"/Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions/results/3_metrics/{pfix}/"
os.makedirs(modDir, exist_ok=True)
os.makedirs(metDir, exist_ok=True)

training(
        train_datasets="",
        data_path=processed_path,
        seed=seed,
        batch_size=128,
        learning_rate=lr,
        num_steps=steps,
        model_dir=modDir,
        metric_dir=metDir,
        fprefix=pfix, 
        purity_col = "purity")

/Users/nv4/miniforge3/envs/scadenPuritiesFullyRandomEnvX86/lib/python3.8/site-packages/rich/live.py:231: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

INFO:tensorflow:Assets written to: /Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions/results/3_train/snCounts_balFalse_thresh0_ran0.5_var0.1_samp20_cells100_seed123_lr0.01_steps10_cran0.5//m256/assets


INFO:tensorflow:Assets written to: /Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions/results/3_train/snCounts_balFalse_thresh0_ran0.5_var0.1_samp20_cells100_seed123_lr0.01_steps10_cran0.5//m256/assets


/Users/nv4/miniforge3/envs/scadenPuritiesFullyRandomEnvX86/lib/python3.8/site-packages/rich/live.py:231: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Saved model


...

/Users/nv4/miniforge3/envs/scadenPuritiesFullyRandomEnvX86/lib/python3.8/site-packages/rich/live.py:231: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Saved model


INFO:tensorflow:Assets written to: /Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions/results/3_train/snCounts_balFalse_thresh0_ran0.5_var0.1_samp20_cells100_seed123_lr0.01_steps10_cran0.5//m1024/assets


INFO:tensorflow:Assets written to: /Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions/results/3_train/snCounts_balFalse_thresh0_ran0.5_var0.1_samp20_cells100_seed123_lr0.01_steps10_cran0.5//m1024/assets


Saved model


In [8]:
predDir = f"/Users/nv4/gitClones/scadenPuritiesFullyRandomSim/testingPurityFunctions/results/4_predict/{pfix}/"
os.makedirs(predDir, exist_ok=True)

prediction(
        model_dir=modDir,
        data_path=BULK_PURITY,
        out_name=os.path.join(predDir, f"{pfix}_Predicted.txt"),
        purity_col="purity")

1/1 [==============================] - 0s 52ms/step
